#warehouse setup

In [0]:
%sql
CREATE CATALOG data_warehouse;
CREATE SCHEMA data_warehouse.bndes;
CREATE VOLUME data_warehouse.bndes.files_ingest;

#load file setup

In [0]:
import os
current_path = os.getcwd()

dbutils.fs.cp(
    current_path+"/files/operacoes-financiamento-operacoes-nao-auto_202601.csv",
    "/Volumes/data_warehouse/bndes/files_ingest/operacoes-financiamento-operacoes-nao-auto_202601.csv"
)
print("File copied successfully")


#job config

In [0]:
%pip install --upgrade databricks-sdk==0.70.0
%restart_python

from databricks.sdk.service.jobs import JobSettings as Job
from databricks.sdk import WorkspaceClient

wsclient = WorkspaceClient()
user_email = wsclient.current_user.me().user_name

DATA_MASTER_PIPELINE = Job.from_dict(
    {
        "name": "DATA_MASTER_PIPELINE_DEMO",
        "trigger": {
            "pause_status": "PAUSED",
            "file_arrival": {
                "url": "/Volumes/data_warehouse/bndes/files_ingest/",
                "min_time_between_triggers_seconds": 60,
                "wait_after_last_change_seconds": 60,
            },
        },
        "tasks": [
            {
                "task_key": "00_FILE_INGEST",
                "notebook_task": {
                    "notebook_path": "00.file_ingest",
                    "source": "GIT",
                },
            },
            {
                "task_key": "01_BRONZE_TABLE",
                "depends_on": [
                    {
                        "task_key": "00_FILE_INGEST",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "01.bronze_table",
                    "source": "GIT",
                },
                "email_notifications": {
                    "on_failure": [
                        user_email,
                    ],
                },
            },
            {
                "task_key": "02_SILVER_TABLE",
                "depends_on": [
                    {
                        "task_key": "01_BRONZE_TABLE",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "02.silver_table",
                    "source": "GIT",
                },
                "email_notifications": {
                    "on_failure": [
                        user_email,
                    ],
                },
            },
            {
                "task_key": "03_GOLD_TABLE",
                "depends_on": [
                    {
                        "task_key": "02_SILVER_TABLE",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "03.gold_table",
                    "source": "GIT",
                },
                "email_notifications": {
                    "on_failure": [
                        user_email,
                    ],
                },
            },
        ],
        "git_source": {
            "git_url": "https://github.com/fernandomiguel99/case_data_master_data_eng.git",
            "git_provider": "gitHub",
            "git_branch": "main",
        },
        "queue": {
            "enabled": True,
        },
        "environments": [
            {
                "environment_key": "PIPELINE_ENV",
                "spec": {
                    "environment_version": "5",
                },
            },
        ],
        "performance_target": "PERFORMANCE_OPTIMIZED",
    }
)

try:
    wsclient.jobs.reset(new_settings=DATA_MASTER_PIPELINE, job_id=87138494386020)
except:
    wsclient.jobs.create(**DATA_MASTER_PIPELINE.as_shallow_dict())
